In [2]:
# Core scverse libraries
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import scanpy.external as sce
import json

In [3]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
    fontsize=22
)
plt.rcParams['axes.grid'] = False

In [3]:
adata = sc.read_h5ad("/Users/takahiro/Desktop/project/Reha/snRNAseq_scanpy/adata/CM_Ctrl_6W.h5ad")

In [4]:
adata.obs["type"] = adata.obs["type"].map({"Ctrl":"Ctrl","Reha6W":"Ex","SED6W":"SED"})

In [118]:
sc.pl.highest_expr_genes(adata, n_top=20)

normalizing counts per cell
    finished (0:00:00)


In [119]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='sample', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='sample', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='sample', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [120]:
sc.pl.violin(
    adata,
    ["pct_counts_mt"],
    jitter=0,
    multi_panel=True,
)

In [121]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

In [122]:
adata.raw = adata.copy()

In [123]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [124]:
sc.pp.log1p(adata)

In [125]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


In [126]:
sc.pl.highly_variable_genes(adata)

In [127]:
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])

regressing out ['total_counts', 'pct_counts_mt']
    sparse input is densified and may lead to high memory use
    finished (0:02:59)


In [128]:
sc.pp.scale(adata, max_value=10)

In [129]:
sc.tl.pca(adata, svd_solver="arpack")

computing PCA
    with n_comps=50
    finished (0:01:01)


In [130]:
sc.pl.pca_variance_ratio(adata, log=True)

In [131]:
adata.write_h5ad("./adata/CM_Ctrl_6W_beforecuration_241025.h5ad")

In [13]:
adata = sc.read_h5ad("./adata/CM_Ctrl_6W_beforecuration_241025.h5ad")

# Over clustering

In [14]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15)

computing neighbors
    using 'X_pca' with n_pcs = 15
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:14)


In [15]:
sc.tl.umap(adata, spread=0.5, min_dist=0.5)

computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:05)


# Downstream

In [16]:
sc.pl.umap(adata, color=["type"], vmax=5)

In [17]:
sc.tl.leiden(adata, resolution=1)

running Leiden clustering


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_95060/1886685787.py:1: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=1)


    finished: found 14 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


In [18]:
sc.pl.umap(adata, color=["leiden"])

In [19]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='leiden', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='leiden', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='leiden', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [20]:
sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3", #CMs
  "Dcn", "Col1a1", "Postn", # FBs
  "Frmd3", "Dlc1", "Myh11", # SMCs
  "Pecam1", "Cdh5", "Vwf","Npr3", # Enforthelial, cardialcells
  "Ptprc", "Mrc1", "Cd163", # Macrophages
  "Cd3e","Skap1", "Cd79a", "Cd79b","Il7r", "Kit", # Tcell Bcell
  "Lmnb1","Slpi","Retnlg","S100a9", #Granulocytes
  "Nrxn1", "Nrxn3", "Upk3b","Msln","Gpc3", 
  "Plin1", "Xkr4", "Acta2","Ms4a1","Ncr1","Vtn","Colec11","Steap4","Kcnj8","Mmrn1","Flt4"
], groupby="leiden", vmax=4)

In [21]:
# cluster 12 13 low quality
adata = adata[~adata.obs["leiden"].isin(["11","12","13"])]

# Down stream

In [22]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15)
sc.tl.umap(adata, spread=0.5, min_dist=0.5)

computing neighbors
    using 'X_pca' with n_pcs = 15
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:03)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:05)


In [23]:
sc.tl.leiden(adata, resolution=0.2)

running Leiden clustering
    finished: found 3 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


In [7]:
sc.settings.set_figure_params(figsize=(6, 6), dpi=100)
sc.pl.umap(adata, color=["type"], save="_CM_sample", title="Sample type")

In [4]:
sc.settings.set_figure_params(figsize=(6, 6), dpi=100)
sc.pl.umap(adata, color=["leiden"],save="_CM_leiden")

In [16]:
import matplotlib.pyplot as plt
import pandas as pd

# 'type'と'leiden'ごとの集計を行い、割合を計算
type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)
type_leiden_percentage = type_leiden_counts.div(type_leiden_counts.sum(axis=1), axis=0) * 100

# 積み上げ棒グラフの作成
type_leiden_percentage.plot(kind='bar', stacked=True, figsize=(5, 4))

# グラフのタイトルとラベルを設定
plt.title('Leiden Proportions per sample')
plt.xlabel('Type')
plt.ylabel('Percentage')

# 凡例を調整
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
# グラフを表示
plt.tight_layout()
plt.show()


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_50392/2379516000.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)


In [191]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:07)


In [192]:
sc.tl.dendrogram(adata,groupby="leiden")

    using 'X_pca' with n_pcs = 50
Storing dendrogram info using `.uns['dendrogram_leiden']`


In [193]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden", standard_scale="var", n_genes=15)

In [194]:
adata.write_h5ad("./adata/CM_Ctrl_6W_analysed.h5ad")

In [3]:
adata = sc.read_h5ad("./adata/CM_Ctrl_6W_analysed.h5ad")

In [14]:
sc.settings.set_figure_params(figsize=(6, 6), dpi=100)
sc.pl.umap(
    adata,
    color="leiden",
    palette=["#D55E00", "#ADD8E6", "#EFC000"],   # ここを変更
    save="_CM_leiden"
)

In [18]:
import matplotlib.pyplot as plt
import pandas as pd

type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)
type_leiden_percentage = type_leiden_counts.div(type_leiden_counts.sum(axis=1), axis=0) * 100

colors = ["#D55E00", "#ADD8E6", "#EFC000"]

ax = type_leiden_percentage.plot(
    kind='bar',
    stacked=True,
    figsize=(5,4),
    color=colors
)

ax.set_xticklabels(["Ctrl","Ex","SED"], rotation=0)

plt.title('Leiden Proportions per sample')
plt.xlabel('Type')
plt.ylabel('Percentage')

plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)

plt.tight_layout()
plt.show()


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_20615/2901097539.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)


In [10]:
adata.obs["type"].value_counts()

type
Ctrl      3935
Reha6W    3897
SED6W     3033
Name: count, dtype: int64

In [8]:
adata_MI = adata[adata.obs["type"]!="Ctrl"]

In [9]:
adata_MI

View of AnnData object with n_obs × n_vars = 6930 × 24225
    obs: 'sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'scrublet', 'type_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [7]:
adata_MI.write_h5ad("./adata/CM_MI_Ctrl_6W_analysed.h5ad")

In [10]:
adata_MI = adata_MI.raw.to_adata().copy()

In [11]:
sc.pl.umap(adata_MI, color=["type","leiden"])

In [9]:
adata_MI.write_h5ad("./adata/CM_MI_Ctrl_6W_analysed_raw.h5ad")

# Intra sample comparison

In [5]:
adata = adata.raw.to_adata().copy()

In [29]:
adata.write_h5ad("./adata/CM_Ctrl_6W_analysed_raw.h5ad")

In [ ]:
adata = sc.read_h5ad("./adata/CM_Ctrl_6W_analysed_raw.h5ad")

In [6]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [7]:
sc.pp.log1p(adata)

In [8]:
print(adata.X)

  (0, 8)	0.6971460289906608
  (0, 10)	0.8153723956594539
  (0, 12)	0.22474501009158065
  (0, 13)	1.8785601211511584
  (0, 14)	0.5630443504006988
  (0, 17)	1.0166875855805892
  (0, 20)	0.22474501009158065
  (0, 21)	1.0166875855805892
  (0, 25)	0.4081327830260533
  (0, 27)	0.4081327830260533
  (0, 33)	0.22474501009158065
  (0, 35)	0.5630443504006988
  (0, 40)	1.1841876155148705
  (0, 43)	1.3276153593428761
  (0, 44)	0.22474501009158065
  (0, 46)	0.9210874346710384
  (0, 53)	0.22474501009158065
  (0, 57)	0.22474501009158065
  (0, 59)	1.4530284127076585
  (0, 60)	0.22474501009158065
  (0, 61)	0.6971460289906608
  (0, 65)	0.4081327830260533
  (0, 66)	1.0166875855805892
  (0, 95)	0.4081327830260533
  (0, 98)	0.6971460289906608
  :	:
  (10864, 23764)	3.1493726191012272
  (10864, 23809)	3.1493726191012272
  (10864, 23816)	6.770256844367929
  (10864, 23833)	3.1493726191012272
  (10864, 23853)	3.1493726191012272
  (10864, 23881)	3.820847124762276
  (10864, 23895)	3.1493726191012272
  (10864, 239

In [9]:
# Obtain cluster-specific differentially expressed gene
sc.tl.rank_genes_groups(adata, groupby="leiden",groups=("2","0"),reference="0", method="wilcoxon")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:09)


In [10]:
# Extract rank genes groups data from adata
rank_genes_groups = adata.uns["rank_genes_groups"]

# Convert the data into a DataFrame
groups = rank_genes_groups['names'].dtype.names  # Get all group names
dfs = []

for group in groups:
    df = pd.DataFrame({
        'gene': rank_genes_groups['names'][group],
        'logfc': rank_genes_groups['logfoldchanges'][group],
        'pvals_adj': rank_genes_groups['pvals_adj'][group]
    })
    df['group'] = group  # Add group information to identify the cell type/cluster
    dfs.append(df)

# Concatenate all groups into a single DataFrame
DEG = pd.concat(dfs, ignore_index=True)

In [11]:
DEG_leiden = DEG

In [12]:
DEG_leiden.to_csv("./DEG_leiden_2vs0.csv")

In [ ]:
DEG = DEG.loc[DEG["pvals_adj"] < 0.1]
DEG_2 = DEG.loc[DEG["logfc"] > 0.25]
DEG_0 = DEG.loc[DEG["logfc"] < -0.25]

In [14]:
DEG_0["logfc"] = -DEG_0["logfc"]

/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_16478/2729650585.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DEG_0["logfc"] = -DEG_0["logfc"]


In [22]:
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
    fontsize=22
)

In [23]:
import matplotlib.pyplot as plt
import numpy as np
from adjustText import adjust_text

# Sort the DataFrame by logfc_6W in descending order
DEG_sorted = DEG_2.sort_values(by='pvals_adj', ascending=True)
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])



# Create the dot plot
plt.figure(figsize=(10, 8))
plt.scatter(range(len(DEG_sorted)), DEG_sorted['neg_log_pvals_adj'], color='blue', s=50)
plt.ylabel('-log10(Adjusted p-value)')
plt.xlabel('Genes (Index)')
plt.title('Rank Plot of -log10(Adjusted p-value)')

# Select the top 10 genes for annotation
top_5_genes = DEG_sorted.head(5)

# Add annotations for the top 5 genes
texts = []
for i, row in top_5_genes.iterrows():
    text = plt.text(i, row['neg_log_pvals_adj'], row['gene'], fontsize=24, color='black')
    texts.append(text)

# Adjust text to avoid overlap
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))

# Set x-ticks to blank
plt.xticks([])

# Display the plot
plt.tight_layout()
plt.show()


# Enrichment analysis

In [25]:
import gseapy as gp
mouse = gp.get_library_name(organism='Mouse')
mouse

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down',
 'Allen_Brain_Atlas_up',
 'Azimuth_2023',
 'Azimuth_Cell_Types_2021',
 'BioCarta_2013',
 'BioCarta_2015',
 'BioCarta_2016',
 'BioPlanet_2019',
 'BioPlex_2017',
 'CCLE_Proteomics_2020',
 'CM4AI_U2OS_Protein_Localization_Assemblies',
 'COMPARTMENTS_Curated_2025',
 'COMPARTMENTS_Experimental_2025',
 'CORUM',
 'COVID-19_Related_Gene_Sets',
 'COVID-19_Related_Gene_Sets_2021',
 'Cancer_Cell_Line_Encyclopedia',
 'Carcinogenome',
 'CellMarker_2024',
 'CellMarker_Augmented_2021',
 'ChEA_2013',
 'ChEA_2015',
 'ChEA_2016',
 'ChEA_2022',
 'Chromosome_Location',
 'Chromosome_Location_hg19',
 'ClinVar_2019',
 'ClinVar_2025',
 'DGIdb_Drug_Targets_2024',
 'DSigDB',
 'Data_Acquisition_Method_Most_Popul

## Cluster 2 DEG

In [36]:
up_genes = DEG_2["gene"]
up_genes = up_genes.squeeze().str.strip().to_list()

In [37]:
geneset_list = ['MSigDB_Hallmark_2020','GO_Biological_Process_2023','KEGG_2019_Mouse','Reactome_2022']
enr = gp.enrichr(gene_list=up_genes,
                 gene_sets=geneset_list,
                 organism='mouse', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )
enr.results['n_genes'] = [int(x.split('/')[0]) for x in enr.results['Overlap']]

metrics_to_sort = '-log10(adjusted P-value)'
for geneset in geneset_list:
    print(geneset)
    df = enr.results[enr.results['Gene_set'] == geneset]
    df = df[df['Adjusted P-value'] < 0.05]
    df["-log10(adjusted P-value)"] = -np.log10(df['Adjusted P-value'])
    df = df.sort_values(metrics_to_sort,ascending=False)
    display(df[:30])
    
    # plot
    n_rank = 10
    plt.rcParams['axes.grid'] = False
    plt.rcParams['figure.figsize'] = 8,6
    plt.barh(width=df[:n_rank][metrics_to_sort],
             y=[x.split(' (')[0] for x in df[:n_rank]['Term']],
             color='darkred')
    plt.gca().invert_yaxis()
    plt.xlabel(metrics_to_sort)
    plt.title(f'{geneset}')
    plt.margins(y=0.02)
    plt.show()

## Cluster 0 DEG

In [39]:
up_genes = DEG_0["gene"]
up_genes = up_genes.squeeze().str.strip().to_list()

In [40]:
geneset_list = ['MSigDB_Hallmark_2020','GO_Biological_Process_2023','KEGG_2019_Mouse','Reactome_2022']
enr = gp.enrichr(gene_list=up_genes,
                 gene_sets=geneset_list,
                 organism='mouse', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir="CM_cluster0_DEG.csv", # don't write to disk
                )
enr.results['n_genes'] = [int(x.split('/')[0]) for x in enr.results['Overlap']]

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(


In [41]:
metrics_to_sort = '-log10(adjusted P-value)'
for geneset in geneset_list:
    print(geneset)
    df = enr.results[enr.results['Gene_set'] == geneset]
    df = df[df['Adjusted P-value'] < 0.05]
    df["-log10(adjusted P-value)"] = -np.log10(df['Adjusted P-value'])
    df = df.sort_values(metrics_to_sort,ascending=False)
    display(df[:30])
    
    # plot
    n_rank = 10
    plt.rcParams['axes.grid'] = False
    plt.rcParams['figure.figsize'] = 8,6
    plt.barh(width=df[:n_rank][metrics_to_sort],
             y=[x.split(' (')[0] for x in df[:n_rank]['Term']],
             color='darkred')
    plt.gca().invert_yaxis()
    plt.xlabel(metrics_to_sort)
    plt.title(f'{geneset}')
    plt.margins(y=0.02)
    plt.show()

MSigDB_Hallmark_2020


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
0,MSigDB_Hallmark_2020,UV Response Dn,49/144,4.635293e-16,2.317647e-14,0,0,5.020165,177.250289,PHF3;CITED2;CELF2;SCHIP1;SERPINE1;PTEN;PTPRM;P...,49,13.634953
1,MSigDB_Hallmark_2020,Mitotic Spindle,49/199,3.165197e-10,7.912993e-09,0,0,3.169726,69.333434,DOCK4;TRIO;WASL;SMC3;ARHGAP5;AKAP13;PCM1;OPHN1...,49,8.101659
2,MSigDB_Hallmark_2020,Myogenesis,45/200,3.243509e-08,4.054386e-07,0,0,2.810210,48.459329,EIF4A2;RB1;ITGB1;MYOM1;ITGB5;FLII;SCHIP1;FHL1;...,45,6.392075
3,MSigDB_Hallmark_2020,heme Metabolism,45/200,3.243509e-08,4.054386e-07,0,0,2.810210,48.459329,SLC22A4;USP15;GYPC;MGST3;NR3C1;CLCN3;ADD1;TRAK...,45,6.392075
4,MSigDB_Hallmark_2020,TGF-beta Signaling,19/54,2.416757e-07,2.416757e-06,0,0,5.216626,79.478790,ACVR1;SMAD1;WWTR1;BMPR2;SMURF2;IFNGR2;SERPINE1...,19,5.616767
5,MSigDB_Hallmark_2020,PI3K/AKT/mTOR Signaling,24/105,3.796177e-05,3.163480e-04,0,0,2.847605,28.985574,ATF1;SMAD2;ACTR3;GSK3B;ACTR2;PRKAA2;CAB39;CAB3...,24,3.499835
6,MSigDB_Hallmark_2020,Hypoxia,32/200,2.337202e-03,1.669430e-02,0,0,1.829571,11.085008,CITED2;GBE1;SERPINE1;NEDD4L;PYGM;RORA;VLDLR;NR...,32,1.777432
7,MSigDB_Hallmark_2020,Androgen Response,18/100,5.845972e-03,3.653732e-02,0,0,2.102812,10.812667,B4GALT1;TSC22D1;HOMER2;GSR;ARID5B;PTPN21;RPS6K...,18,1.437263
8,MSigDB_Hallmark_2020,IL-2/STAT5 Signaling,30/199,7.492103e-03,4.162279e-02,0,0,1.703154,8.335076,CYFIP1;PHTF2;BMPR2;CSF1;RORA;IKZF2;HK2;IGF1R;N...,30,1.380669
9,MSigDB_Hallmark_2020,Protein Secretion,17/96,8.556885e-03,4.278443e-02,0,0,2.060655,9.810816,RAB2A;VPS4B;SNAP23;ADAM10;AP2B1;CLCN3;IGF2R;TO...,17,1.368714


GO_Biological_Process_2023


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
50,GO_Biological_Process_2023,Protein Phosphorylation (GO:0006468),102/500,6.262357e-14,2.508074e-10,0,0,2.524736,76.756087,ATF2;IGF1R;RPS6KA3;PPP4R1;PDK4;MAP3K7;PDK1;ACV...,102,9.600660
51,GO_Biological_Process_2023,Protein Modification Process (GO:0036211),130/711,1.323309e-13,2.649926e-10,0,0,2.216019,65.712656,UBE2D3;ARAF;PLOD2;PPP2R2A;PARK7;PPP4R1;PDK4;LA...,130,9.576766
52,GO_Biological_Process_2023,Phosphorylation (GO:0016310),86/429,1.585684e-11,2.116888e-08,0,0,2.455843,61.070477,GSK3B;SMG1;DGKD;BMPR2;MAST2;PIK3CB;LIMD1;HK2;R...,86,7.674302
53,GO_Biological_Process_2023,Actomyosin Structure Organization (GO:0031032),28/77,1.484164e-10,1.486020e-07,0,0,5.513324,124.772022,FHOD3;ITGB1;ITGB5;ROCK1;FLII;LMOD3;MYOM2;LMOD2...,28,6.827975
54,GO_Biological_Process_2023,Protein Ubiquitination (GO:0016567),83/434,4.131970e-10,3.309708e-07,0,0,2.311285,49.940157,TRIM72;DET1;RNF13;UBE3C;FBH1;UBE2D3;UBE2D1;PRP...,83,6.480210
55,GO_Biological_Process_2023,Positive Regulation Of DNA-templated Transcrip...,182/1243,1.225487e-09,8.180126e-07,0,0,1.702375,34.932617,ATF1;ATF2;CCNT2;CRTC3;ICE1;KDM1A;HNRNPU;RORA;P...,182,6.087240
56,GO_Biological_Process_2023,Ubiquitin-Dependent Protein Catabolic Process ...,71/367,4.594447e-09,2.628680e-06,0,0,2.336321,44.853657,GSK3B;TRIM72;UBE3C;RNF13;USP33;UBE2D3;OTUD7B;U...,71,5.580262
57,GO_Biological_Process_2023,Positive Regulation Of Transcription By RNA Po...,141/938,1.977386e-08,9.899290e-06,0,0,1.741359,30.889810,ATF1;ATF2;CCNT2;CRTC3;KDM1A;ZMYND8;HNRNPU;RORA...,141,5.004396
58,GO_Biological_Process_2023,Peptidyl-Serine Phosphorylation (GO:0018105),38/158,5.881537e-08,2.617284e-05,0,0,3.059636,50.939466,GSK3B;SMG1;CAMK2D;CAB39;ROCK1;TNKS;MAST2;RPS6K...,38,4.582149
59,GO_Biological_Process_2023,Myofibril Assembly (GO:0030239),18/46,7.607652e-08,3.046865e-05,0,0,6.176692,101.245404,FHOD3;TMOD1;ADPRHL1;ACTN2;FLII;LMOD3;MYOM2;KLH...,18,4.516147


KEGG_2019_Mouse


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
4055,KEGG_2019_Mouse,Adherens junction,22/72,4.719499e-07,0.000132,0,0,4.231454,61.637028,SMAD2;YES1;INSR;PTPRM;LMO7;WASL;IQGAP1;SORBS1;...,22,3.880500
4056,KEGG_2019_Mouse,Circadian rhythm,13/30,1.245072e-06,0.000174,0,0,7.332418,99.693886,PRKAB2;PRKAA2;FBXW11;BHLHE41;CUL1;RORA;CSNK1D;...,13,3.760231
4057,KEGG_2019_Mouse,Renal cell carcinoma,20/68,3.011606e-06,0.000245,0,0,4.003238,50.893307,EGLN1;TGFB2;JUN;ARAF;GAB1;ARNT;PTPN11;PIK3CB;H...,20,3.610940
4058,KEGG_2019_Mouse,Autophagy,30/130,3.511684e-06,0.000245,0,0,2.889406,36.289250,RAB7;MTMR3;PRKAA2;ITPR1;PTEN;PIK3CB;AMBRA1;HIF...,30,3.610940
4059,KEGG_2019_Mouse,Proteoglycans in cancer,40/203,6.468514e-06,0.000361,0,0,2.367920,28.293244,ITGB1;DDX5;CAMK2D;ITGB5;ROCK1;ARAF;ITPR1;ITPR2...,40,3.442561
4060,KEGG_2019_Mouse,Focal adhesion,39/199,9.587295e-06,0.000363,0,0,2.351139,27.167581,ITGB1;GSK3B;ITGB5;ROCK1;PTEN;PIK3CB;LAMC1;ARHG...,39,3.440110
4061,KEGG_2019_Mouse,Insulin resistance,26/110,9.718278e-06,0.000363,0,0,2.977411,34.363793,GSK3B;PRKAA2;PTEN;PPP1R3A;PYGM;PIK3CB;ACACB;FO...,26,3.440110
4062,KEGG_2019_Mouse,Axon guidance,36/180,1.304647e-05,0.000363,0,0,2.409689,27.101752,SEMA5A;ITGB1;GSK3B;CAMK2D;BMPR2;ROCK1;LRRC4;PI...,36,3.440110
4063,KEGG_2019_Mouse,Regulation of actin cytoskeleton,41/217,1.445321e-05,0.000363,0,0,2.247422,25.046607,CHRM2;ITGB1;CYFIP1;NCKAP1;ITGB5;ROCK1;ARAF;IQG...,41,3.440110
4064,KEGG_2019_Mouse,Insulin signaling pathway,30/139,1.449639e-05,0.000363,0,0,2.649506,29.519765,GSK3B;PRKAA2;ARAF;CBLB;PPP1R3A;PYGM;PIK3CB;ACA...,30,3.440110


Reactome_2022


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
4334,Reactome_2022,Signaling By Rho GTPases R-HSA-194315,124/644,1.053811e-14,7.040720e-12,0,0,2.361911,76.015226,ITSN2;CYFIP1;NCKAP1;TRIO;FAM13B;KDM1A;WIPF3;FA...,124,11.152383
4335,Reactome_2022,"Signaling By Rho GTPases, Miro GTPases And RHO...",126/660,1.233284e-14,7.040720e-12,0,0,2.337858,74.873439,ITSN2;CYFIP1;NCKAP1;TRIO;FAM13B;KDM1A;WIPF3;FA...,126,11.152383
4336,Reactome_2022,RHO GTPase Cycle R-HSA-9012999,95/441,1.468857e-14,7.040720e-12,0,0,2.702288,86.072489,ITSN2;ITGB1;CYFIP1;NCKAP1;DOCK4;TRIO;ABCD3;FAM...,95,11.152383
4337,Reactome_2022,Signal Transduction R-HSA-162582,335/2465,9.878450e-13,3.551303e-10,0,0,1.606056,44.396596,ITSN2;NCKAP1;CYFIP1;TRIO;ZFYVE9;WIPF3;SERPINE1...,335,9.449612
4338,Reactome_2022,RAC1 GTPase Cycle R-HSA-9013149,48/178,1.568700e-11,4.511581e-09,0,0,3.584790,89.183081,ITGB1;CYFIP1;NCKAP1;DOCK4;TRIO;FAM13B;FAM13A;W...,48,8.345671
4339,Reactome_2022,Circadian Clock R-HSA-400253,24/69,8.418702e-09,1.746493e-06,0,0,5.135929,95.491351,CPT1A;MEF2C;CRTC3;CHD9;NCOA6;SERPINE1;BHLHE41;...,24,5.757833
4340,Reactome_2022,Diseases Of Signal Transduction By Growth Fact...,78/424,8.501704e-09,1.746493e-06,0,0,2.198008,40.845578,GSK3B;HIP1;ZFYVE9;ARAF;PTEN;GCC2;PIK3CB;TENT4A...,78,5.757833
4341,Reactome_2022,RAC3 GTPase Cycle R-HSA-9013423,27/93,8.010742e-08,1.439931e-05,0,0,3.941215,64.399049,ITGB1;CYFIP1;NCKAP1;TRIO;SNAP23;ARHGAP5;ARHGAP...,27,4.841658
4342,Reactome_2022,RHOC GTPase Cycle R-HSA-9013106,23/73,1.394538e-07,2.228161e-05,0,0,4.426151,69.869158,STARD13;ABCD3;ARHGEF12;ROCK1;JUP;ERBIN;MACO1;M...,23,4.652053
4343,Reactome_2022,RHOQ GTPase Cycle R-HSA-9013406,20/59,2.364623e-07,3.285072e-05,0,0,4.929518,75.212006,JUP;SNAP23;WASL;IQGAP1;CDC42BPB;ARHGAP5;ARHGAP...,20,4.483455
